# re 정규표현식 간단 실습

## 실습 목표

이 노트북은 Python의 `re` 모듈을 사용하여 정규표현식의 기본 사용법을 익히는 간단 실습입니다.

정규표현식은 문자열에서 특정 패턴을 **찾고**, **추출하고**, **검증하고**, **치환**할 때 사용합니다.

예를 들어 다음과 같은 작업에 활용할 수 있습니다.

- 이메일 주소 찾기
- 전화번호 찾기
- 날짜 형식 찾기
- 숫자만 추출하기
- 불필요한 공백 제거하기
- 문자열 패턴 검증하기
- 개인정보 마스킹하기

---

## 실습 구성

1. `re` 모듈 불러오기
2. `search`, `match`, `fullmatch` 차이
3. `findall`, `finditer`로 여러 패턴 찾기
4. 문자 클래스와 반복 기호
5. 그룹 추출
6. `sub`, `split`으로 문자열 정리
7. 이메일, 전화번호, 날짜 추출 실습
8. Pandas와 함께 사용하는 정규표현식
9. 종합 실습 문제

# 1. re 모듈 불러오기

Python에서 정규표현식을 사용하려면 표준 라이브러리인 `re` 모듈을 불러옵니다.

정규표현식을 작성할 때는 보통 문자열 앞에 `r`을 붙입니다.

예: `r"\d+"`

여기서 `r`은 raw string을 의미합니다. 백슬래시(`\`)를 정규표현식 기호로 편하게 사용하기 위해 자주 사용합니다.

In [ ]:
import re

text = "주문번호 A-2026-001, 고객 전화번호는 010-1234-5678입니다."

print(text)

# 2. 기본 함수: search, match, fullmatch

## `re.search()`

문자열 전체에서 패턴이 **어디든 존재하는지** 찾습니다.

## `re.match()`

문자열의 **시작 부분**에서 패턴이 일치하는지 확인합니다.

## `re.fullmatch()`

문자열 전체가 패턴과 **완전히 일치하는지** 확인합니다.

In [ ]:
sample = "ABC-12345"

# 패턴 의미:
# [A-Z]+  : 대문자 알파벳이 1개 이상
# -       : 하이픈 문자
# \d+     : 숫자가 1개 이상
pattern = r"[A-Z]+-\d+"

print("search:", re.search(pattern, sample))
print("match:", re.match(pattern, sample))
print("fullmatch:", re.fullmatch(pattern, sample))

In [ ]:
# 같은 패턴을 다른 문자열에 적용해 봅니다.
# 문자열 앞에 다른 글자가 붙어 있으므로 match와 fullmatch 결과가 달라집니다.

sample2 = "주문번호 ABC-12345 확인"

print("search:", re.search(pattern, sample2))
print("match:", re.match(pattern, sample2))
print("fullmatch:", re.fullmatch(pattern, sample2))

## 정리

| 함수 | 의미 | 사용 상황 |
|---|---|---|
| `search` | 문자열 어딘가에 패턴이 있는지 확인 | 문서 안에서 찾기 |
| `match` | 문자열 시작 부분이 패턴과 맞는지 확인 | 접두 형식 확인 |
| `fullmatch` | 문자열 전체가 패턴과 맞는지 확인 | 입력값 형식 검증 |

# 3. 여러 개 찾기: findall, finditer

## `re.findall()`

패턴과 일치하는 모든 문자열을 리스트로 반환합니다.

## `re.finditer()`

패턴과 일치하는 결과를 반복 가능한 match 객체로 반환합니다.  
각 결과의 위치(`start`, `end`)까지 확인할 수 있습니다.

In [ ]:
log_text = """
2026-01-01 주문 3건, 매출 120000원
2026-01-02 주문 5건, 매출 235000원
2026-01-03 주문 2건, 매출 98000원
"""

# 숫자가 1개 이상 반복되는 패턴
number_pattern = r"\d+"

numbers = re.findall(number_pattern, log_text)
print(numbers)

In [ ]:
# finditer는 찾은 값의 위치까지 확인할 수 있습니다.
for match in re.finditer(number_pattern, log_text):
    print("값:", match.group(), "시작 위치:", match.start(), "끝 위치:", match.end())

# 4. 정규표현식 기본 기호

자주 사용하는 기호는 다음과 같습니다.

| 기호 | 의미 | 예시 |
|---|---|---|
| `.` | 임의의 한 글자 | `a.c` |
| `\d` | 숫자 | `\d+` |
| `\D` | 숫자가 아닌 문자 | `\D+` |
| `\w` | 문자, 숫자, 밑줄 | `\w+` |
| `\s` | 공백 문자 | `\s+` |
| `[]` | 문자 집합 | `[가-힣]+`, `[A-Z]+` |
| `[^]` | 제외 문자 집합 | `[^0-9]+` |
| `*` | 0번 이상 반복 | `a*` |
| `+` | 1번 이상 반복 | `a+` |
| `?` | 0번 또는 1번 | `a?` |
| `{m,n}` | m번 이상 n번 이하 | `\d{2,4}` |
| `^` | 문자열 시작 | `^ABC` |
| `$` | 문자열 끝 | `xyz$` |

In [ ]:
sample = "상품코드: CP-2026-A100, 가격: 35,000원, 수량: 12개"

patterns = {
    "숫자 묶음": r"\d+",
    "영문 대문자 묶음": r"[A-Z]+",
    "한글 묶음": r"[가-힣]+",
    "공백 묶음": r"\s+",
    "상품코드 형태": r"[A-Z]{2}-\d{4}-[A-Z]\d{3}"
}

for name, p in patterns.items():
    print(name, "=>", re.findall(p, sample))

# 5. 그룹 추출

괄호 `()`를 사용하면 정규표현식에서 원하는 부분을 그룹으로 추출할 수 있습니다.

예를 들어 날짜 `2026-05-31`에서 연, 월, 일을 각각 추출할 수 있습니다.

In [ ]:
date_text = "배송 예정일은 2026-05-31입니다."

# 그룹 1: 연도, 그룹 2: 월, 그룹 3: 일
date_pattern = r"(\d{4})-(\d{2})-(\d{2})"

match = re.search(date_pattern, date_text)

if match:
    print("전체 매칭:", match.group(0))
    print("연도:", match.group(1))
    print("월:", match.group(2))
    print("일:", match.group(3))

In [ ]:
# 이름 있는 그룹을 사용하면 결과를 더 읽기 쉽게 만들 수 있습니다.

named_date_pattern = r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})"

match = re.search(named_date_pattern, date_text)

if match:
    print(match.groupdict())
    print("year:", match.group("year"))
    print("month:", match.group("month"))
    print("day:", match.group("day"))

# 6. 문자열 치환: re.sub

`re.sub()`은 정규표현식 패턴에 맞는 문자열을 다른 문자열로 바꿉니다.

실무에서는 다음과 같은 작업에 자주 사용합니다.

- 전화번호 마스킹
- 여러 개의 공백을 하나로 정리
- 특수문자 제거
- 개인정보 비식별화

In [ ]:
message = "고객 홍길동님의 전화번호는 010-1234-5678이고, 보조번호는 02-987-6543입니다."

# 전화번호 패턴을 찾아 가운데/끝 번호를 마스킹합니다.
phone_pattern = r"(\d{2,3})-(\d{3,4})-(\d{4})"
masked = re.sub(phone_pattern, r"-****-****", message)

print("원본:", message)
print("마스킹:", masked)

In [ ]:
# 여러 개의 공백을 하나로 정리하기
messy_text = "상품명:    무선청소기     가격:   129000원     배송: 빠름"

clean_text = re.sub(r"\s+", " ", messy_text).strip()

print("원본:", messy_text)
print("정리:", clean_text)

# 7. 문자열 분리: re.split

`re.split()`은 여러 구분자를 기준으로 문자열을 나눌 때 유용합니다.

예를 들어 쉼표, 세미콜론, 공백이 섞여 있는 문자열을 한 번에 분리할 수 있습니다.

In [ ]:
items = "사과,바나나;포도  딸기|수박"

# 쉼표, 세미콜론, 공백, 파이프 기호 중 하나 이상을 구분자로 사용합니다.
split_pattern = r"[,;\s|]+"

result = re.split(split_pattern, items)
print(result)

# 8. 이메일, 전화번호, 날짜 추출 실습

다음 문장에서 이메일, 전화번호, 날짜를 각각 추출합니다.

In [ ]:
contact_text = """
고객명: 김민수, 이메일: minsu.kim@example.com, 전화: 010-1111-2222, 가입일: 2026-01-15
고객명: 이서연, 이메일: seoyeon_lee@shop.co.kr, 전화: 02-333-4444, 가입일: 2026-02-03
고객명: Park Joon, 이메일: park.joon99@test.org, 전화: 031-555-7777, 가입일: 2026-03-21
"""

email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
phone_pattern = r"\d{2,3}-\d{3,4}-\d{4}"
date_pattern = r"\d{4}-\d{2}-\d{2}"

emails = re.findall(email_pattern, contact_text)
phones = re.findall(phone_pattern, contact_text)
dates = re.findall(date_pattern, contact_text)

print("이메일:", emails)
print("전화번호:", phones)
print("날짜:", dates)

# 9. compile 사용하기

같은 패턴을 여러 번 사용할 때는 `re.compile()`로 패턴 객체를 만들어 두면 편리합니다.

코드 가독성이 좋아지고, 동일한 정규표현식을 반복해서 사용할 때 관리하기 쉽습니다.

In [ ]:
email_regex = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")

texts = [
    "문의: help@example.com",
    "이 문장에는 이메일이 없습니다.",
    "담당자: manager@test.co.kr"
]

for t in texts:
    match = email_regex.search(t)
    if match:
        print("이메일 발견:", match.group())
    else:
        print("이메일 없음")

# 10. Pandas와 정규표현식 함께 사용하기

Pandas의 문자열 메서드에서도 정규표현식을 사용할 수 있습니다.

대표적으로 다음 메서드가 자주 사용됩니다.

| 메서드 | 의미 |
|---|---|
| `str.contains()` | 패턴 포함 여부 확인 |
| `str.extract()` | 패턴에서 그룹 추출 |
| `str.replace()` | 패턴 치환 |
| `str.split()` | 패턴 기준 분리 |

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": ["CP-2026-0001", "CP-2026-0002", "RT-2026-0003", "CP-2026-0004"],
    "customer_text": [
        "김민수 / 010-1111-2222 / minsu@example.com",
        "이서연 / 010-2222-3333 / seoyeon@test.co.kr",
        "박준호 / 02-333-4444 / junho@test.org",
        "최유진 / 번호없음 / yujin@example.com"
    ],
    "memo": [
        "빠른 배송 요청",
        "주소 확인 필요",
        "반품 접수",
        "VIP 고객, 선물 포장"
    ]
})

orders

In [ ]:
# ------------------------------------------------------------
# str.contains(): 특정 패턴 포함 여부 확인
# ------------------------------------------------------------
# 주문번호가 CP로 시작하는지 확인합니다.

orders["is_cp_order"] = orders["order_id"].str.contains(r"^CP", regex=True)
orders

In [ ]:
# ------------------------------------------------------------
# str.extract(): 그룹 추출
# ------------------------------------------------------------
# customer_text에서 전화번호를 추출합니다.
# 전화번호가 없는 행은 NaN으로 표시됩니다.

orders["phone"] = orders["customer_text"].str.extract(r"(\d{2,3}-\d{3,4}-\d{4})")
orders

In [ ]:
# 이메일 추출
orders["email"] = orders["customer_text"].str.extract(r"([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})")
orders

In [ ]:
# ------------------------------------------------------------
# str.replace(): 개인정보 마스킹
# ------------------------------------------------------------
# 전화번호를 마스킹합니다.

orders["customer_text_masked"] = orders["customer_text"].str.replace(
    r"(\d{2,3})-(\d{3,4})-(\d{4})",
    r"-****-****",
    regex=True
)

orders[["customer_text", "customer_text_masked"]]

# 11. 종합 실습: 주문 로그에서 정보 추출하기

아래 로그 문자열에서 다음 정보를 추출해 보겠습니다.

- 날짜
- 주문번호
- 상태값
- 금액

예시 로그:

`[2026-01-01] ORDER=CP-2026-0001 STATUS=PAID AMOUNT=129000`

In [ ]:
log_lines = [
    "[2026-01-01] ORDER=CP-2026-0001 STATUS=PAID AMOUNT=129000",
    "[2026-01-02] ORDER=CP-2026-0002 STATUS=CANCEL AMOUNT=0",
    "[2026-01-03] ORDER=RT-2026-0003 STATUS=RETURN AMOUNT=45000",
    "[2026-01-04] ORDER=CP-2026-0004 STATUS=PAID AMOUNT=78000",
]

log_pattern = re.compile(
    r"\[(?P<date>\d{4}-\d{2}-\d{2})\]\s+"
    r"ORDER=(?P<order_id>[A-Z]{2}-\d{4}-\d{4})\s+"
    r"STATUS=(?P<status>[A-Z]+)\s+"
    r"AMOUNT=(?P<amount>\d+)"
)

parsed_rows = []

for line in log_lines:
    match = log_pattern.search(line)
    if match:
        parsed_rows.append(match.groupdict())

log_df = pd.DataFrame(parsed_rows)
log_df["amount"] = log_df["amount"].astype(int)
log_df["date"] = pd.to_datetime(log_df["date"])

log_df

In [ ]:
# 상태별 주문 수와 금액 합계 확인
summary = log_df.groupby("status").agg(
    order_count=("order_id", "count"),
    total_amount=("amount", "sum")
)

summary

# 12. 실습 문제

아래 문제를 직접 풀어보세요.

## 문제 1
다음 문자열에서 모든 숫자를 추출하세요.

```python
text = "상품 A는 12개, 상품 B는 3개, 상품 C는 105개입니다."
```

## 문제 2
다음 문자열에서 이메일만 추출하세요.

```python
text = "문의는 help@test.com 또는 admin@example.co.kr 로 보내주세요."
```

## 문제 3
다음 전화번호를 마스킹하세요.

```python
text = "연락처: 010-9876-5432"
```

결과 예시:

```text
연락처: 010-****-****
```

## 문제 4
다음 주문번호 중 `CP-연도-4자리번호` 형식에 맞는 것만 찾으세요.

```python
order_ids = ["CP-2026-0001", "C-2026-01", "RT-2026-0002", "CP-2025-1234"]
```

## 문제 5
Pandas DataFrame에서 이메일 도메인만 추출해 보세요.

# 13. 실습 문제 풀이 예시

In [ ]:
# 문제 1 풀이
text = "상품 A는 12개, 상품 B는 3개, 상품 C는 105개입니다."
print(re.findall(r"\d+", text))

In [ ]:
# 문제 2 풀이
text = "문의는 help@test.com 또는 admin@example.co.kr 로 보내주세요."
print(re.findall(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text))

In [ ]:
# 문제 3 풀이
text = "연락처: 010-9876-5432"
masked = re.sub(r"(\d{2,3})-(\d{3,4})-(\d{4})", r"-****-****", text)
print(masked)

In [ ]:
# 문제 4 풀이
order_ids = ["CP-2026-0001", "C-2026-01", "RT-2026-0002", "CP-2025-1234"]
valid_pattern = r"CP-\d{4}-\d{4}"

valid_ids = [oid for oid in order_ids if re.fullmatch(valid_pattern, oid)]
print(valid_ids)

In [ ]:
# 문제 5 풀이
email_df = pd.DataFrame({
    "email": ["help@test.com", "admin@example.co.kr", "user123@shop.net"]
})

# @ 뒤의 도메인 부분만 추출합니다.
email_df["domain"] = email_df["email"].str.extract(r"@(.+)$")
email_df

# 14. 마무리 정리

이번 실습의 핵심은 다음과 같습니다.

| 기능 | 대표 함수 |
|---|---|
| 패턴 존재 확인 | `re.search`, `re.match`, `re.fullmatch` |
| 여러 개 찾기 | `re.findall`, `re.finditer` |
| 일부 정보 추출 | `group`, `groupdict`, `str.extract` |
| 문자열 치환 | `re.sub`, `str.replace` |
| 문자열 분리 | `re.split`, `str.split` |
| 반복 사용 | `re.compile` |

정규표현식은 처음에는 복잡해 보이지만, 실무에서는 다음 세 가지 작업부터 익히면 충분히 활용할 수 있습니다.

1. 원하는 패턴 찾기
2. 필요한 부분만 추출하기
3. 민감 정보 또는 불필요한 문자열 치환하기